In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("Data/errors_indicators.csv")

C:\Users\ignacio.delatorre\AppData\Local\Temp\ipykernel_31996\4049374003.py:1: DtypeWarning: Columns (1,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Data/errors_indicators.csv")


In [3]:
df_corr = df[['query', 'MAE','spearman_corr']]
df_corr_ext = pd.get_dummies(
    df_corr,
    columns=["query"],
    dtype="int8"
)

df_corr_ext.columns = df_corr_ext.columns.str.replace("^query_", "", regex=True)

In [7]:
df_corr_ext['corr_index'] = (1 - df_corr_ext['spearman_corr']) /2

### Predicción

##### Sobre error

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import sys

In [ ]:
from sklearn.ensemble import RandomForestRegressor

target = "corr_index"
features = df_corr_ext.drop([target, 'spearman_corr'], axis=1).columns

X = df_corr_ext[features]
y = df_corr_ext[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# model
model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# metrics
r2_train = r2_score(y_train, train_pred)
r2_test = r2_score(y_test, test_pred)

mae_train = mean_absolute_error(y_train, train_pred)
mae_test = mean_absolute_error(y_test, test_pred)

# importances
imp_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print('R^2 train: ', r2_train)
print('R^2 test: ', r2_test)
print('MAE train:', mae_train)
print('MAE test:', mae_test)

print(imp_df)

R^2 train:  0.35624894033681453
R^2 test:  0.09742702876623566
MAE train: 0.10505867123350629
MAE test: 0.1259968148565426
                                       feature  importance
0                                          MAE    0.742023
152  fertilizer-crop input-output coefficients    0.023705
212                           land leaf shares    0.013228
262                     resource supply curves    0.012354
189          input-output coefficients by tech    0.012115
..                                         ...         ...
273                      total climate forcing    0.000000
222                   net terrestrial C uptake    0.000000
238                 price of a specific market    0.000000
271                supply of a specific market    0.000000
2                           CO2 concentrations    0.000000

[312 rows x 2 columns]


In [ ]:
from xgboost import XGBRegressor
target = "corr_index"
features = df_corr_ext.drop([target, 'spearman_corr'], axis = 1).columns


X = df_corr_ext[features]
y = df_corr_ext[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. model
model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=10,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=10,
    gamma=1,

    reg_alpha=0.1,
    reg_lambda=1.0,

    n_jobs=-1,
    random_state=42
)
model.fit(X_train, y_train)

# 4. metrics
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

r2_train = r2_score(y_train, train_pred)
r2_test = r2_score(y_test, test_pred)


mae_train = mean_absolute_error(y_train, train_pred)
mae_test = mean_absolute_error(y_test, test_pred)

importances = model.feature_importances_


imp_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print('R^2 train: ',r2_train)
print('R^2 test: ',r2_test)
print('MAE train:',mae_train)
print('MAE test:',mae_test)
print(imp_df)


R^2 train:  0.13475931585749634
R^2 test:  0.13317658505727592
MAE train: 0.1281533465331974
MAE test: 0.1286541488392673
                                       feature  importance
152  fertilizer-crop input-output coefficients    0.054946
39                       ag tech variable cost    0.039444
212                           land leaf shares    0.038978
98                     costs by tech and input    0.037509
262                     resource supply curves    0.037196
..                                         ...         ...
22                            National Account    0.000000
17                GDP per capita PPP by region    0.000000
1                 Basin level available runoff    0.000000
305                water withdrawals by region    0.000000
306                water withdrawals by sector    0.000000

[312 rows x 2 columns]


##### Interpretación

In [38]:
import statsmodels.api as sm


target = "corr_index"
features = df_corr_ext.drop([target, 'spearman_corr'], axis = 1).columns
y = np.log1p(df_corr_ext[target])
X = df_corr_ext[features]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             corr_index   R-squared:                       0.111
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     626.6
Date:                Fri, 17 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:29:32   Log-Likelihood:             8.6730e+05
No. Observations:             1566410   AIC:                        -1.734e+06
Df Residuals:                 1566098   BIC:                        -1.730e+06
Df Model:                         311                                         
Covariance Type:            nonrobust                                         
                                                                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

### Otra óptica

In [40]:
df_corr = df[['query', 'MAE','spearman_corr']]
